In [1]:
with open("verdict.txt","r",encoding="utf-8") as f:
    raw_text = f.read()

print("Total no of charecters:",len(raw_text))
print(raw_text[:99])

Total no of charecters: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [2]:
import re
text = "This is an example text to tokenise."
result = re.split(r'([,.:;?_!"()\']|--|\s)',text)
result = [item.strip() for item in result if item.strip()]
print(result)

['This', 'is', 'an', 'example', 'text', 'to', 'tokenise', '.']


In [3]:
preProcessed =  re.split(r'([,.:;?_!"()\']|--|\s)',raw_text)
preProcessed = [item.strip() for item in preProcessed if item.strip()]
print("Total no of token generated:",len(preProcessed))
print(preProcessed[:10])

Total no of token generated: 4690
['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius']


In [4]:
all_words = sorted(set(preProcessed))
vocab_size = len(all_words)
print("No of unique Tokens:",vocab_size)

No of unique Tokens: 1130


In [5]:
vocab = {token:integer for integer,token in enumerate(all_words)}

for i,item in enumerate(vocab.items()):
    print(item)
    if i>=10:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)


In [6]:
class SimpleTokenizerV1:
    def __init__(self,vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}
    
    def encode(self,text):
        preProcessed =  re.split(r'([,.:;?_!"()\']|--|\s)',text)
        preProcessed = [item.strip() for item in preProcessed if item.strip()]
        ids = [self.str_to_int[s] for s in preProcessed]
        return ids
    
    def decode(self,ids):
        text = " ".join([self.int_to_str[id] for id in ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [7]:
tokenizer = SimpleTokenizerV1(vocab)
encoding_text = """"It's the last he painted, you know,"
 Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(encoding_text)
print(ids)
decoded_text = tokenizer.decode(ids)
print(decoded_text)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]
" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


In [8]:
all_tokens = sorted(list(set(preProcessed)))
all_tokens.extend(["<|endoftext|>","<|unk|>"])
vocabV2 = {token:integer for integer,token in enumerate(all_tokens)}

print(len(vocabV2.items()))

1132


In [9]:
for i,item in enumerate(list(vocabV2.items())[-5:]):
    print(item)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


In [10]:
class SimpleTokenizerV2:
    def __init__(self,vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}
    
    def encode(self,text):
        preProcessed =  re.split(r'([,.:;?_!"()\']|--|\s)',text)
        preProcessed = [item.strip() for item in preProcessed if item.strip()]
        preProcessed = [item if item in self.str_to_int else "<|unk|>" for item in preProcessed]
        ids = [self.str_to_int[s] for s in preProcessed]
        return ids
        
    def decode(self,ids):
        text = " ".join([self.int_to_str[id] for id in ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [11]:
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
encoding_textV2 = " <|endoftext|> ".join((text1, text2))
print(encoding_textV2)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [12]:
tokenizerV2 = SimpleTokenizerV2(vocabV2)
print(tokenizerV2.decode(tokenizerV2.encode(encoding_textV2)))

<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.


In [13]:
!pip install tiktoken


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
from importlib.metadata import version
import tiktoken

In [15]:
print(version("tiktoken"))

0.12.0


In [16]:
tokenizerTK = tiktoken.get_encoding("gpt2")

In [17]:
text = (
 "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
 "of some unknownPlace."
)
integers = tokenizerTK.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 6439, 27271, 13]


In [18]:
strings = tokenizerTK.decode(integers)
print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terracesof some unknownPlace.


In [19]:
enc_text_TK = tokenizerTK.encode(raw_text)
print(len(enc_text_TK))

5145


In [20]:
enc_sample = enc_text_TK[:50]
context_size = 4
for i in range(1,context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(tokenizerTK.decode(context)," -------------> ",tokenizerTK.decode([desired]))

I  ------------->   H
I H  ------------->  AD
I HAD  ------------->   always
I HAD always  ------------->   thought


In [21]:
import torch

In [22]:
from torch.utils.data import DataLoader, Dataset

In [23]:
class GPTDatasetV1(Dataset):
    def __init__(self,text,tokenizer,max_length,stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(text)

        for i in range(0,(len(token_ids) - max_length),stride):
            input_chunk = token_ids[i:i+max_length]
            target_chunk = token_ids[i+1:i+max_length+1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))
    def __len__(self):
        return len(self.input_ids)
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [24]:
def create_dataloader_v1(txt, batch_size=4, max_length=256,stride=128, shuffle=True, drop_last=True,num_workers=0):
    
    tokenizer = tiktoken.get_encoding("gpt2")

    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(dataset,batch_size=batch_size,shuffle=shuffle,drop_last=drop_last,num_workers=num_workers)

    return dataloader

In [30]:
dataloader = create_dataloader_v1(raw_text,8,4,2,False)
dataiter = iter(dataloader)
inputs,targets = next(dataiter)
print("Inputs:",inputs)
print("Targets:",targets)

Inputs: tensor([[   40,   367,  2885,  1464],
        [ 2885,  1464,  1807,  3619],
        [ 1807,  3619,   402,   271],
        [  402,   271, 10899,  2138],
        [10899,  2138,   257,  7026],
        [  257,  7026, 15632,   438],
        [15632,   438,  2016,   257],
        [ 2016,   257,   922,  5891]])
Targets: tensor([[  367,  2885,  1464,  1807],
        [ 1464,  1807,  3619,   402],
        [ 3619,   402,   271, 10899],
        [  271, 10899,  2138,   257],
        [ 2138,   257,  7026, 15632],
        [ 7026, 15632,   438,  2016],
        [  438,  2016,   257,   922],
        [  257,   922,  5891,  1576]])


In [35]:
ex_vocab_size = 6
ex_output_dim = 3

torch.manual_seed(123)
ex_embedding_layer = torch.nn.Embedding(ex_vocab_size, ex_output_dim)
print(ex_embedding_layer.weight)

print("\n-------------------------------\n")

ex_input = torch.tensor([2,3,1,4])
print(ex_embedding_layer(ex_input))

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)

-------------------------------

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [ 0.9178,  1.5810,  1.3010],
        [-1.1589,  0.3255, -0.6315]], grad_fn=<EmbeddingBackward0>)


In [38]:
vocab_size = 50257
output_dim = 256

embedding_layer = torch.nn.Embedding(vocab_size,output_dim)

token_embeddings = embedding_layer(inputs)
print(token_embeddings.shape)

torch.Size([8, 4, 256])


In [39]:
context_length = 4
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(context_length))
print(pos_embeddings.shape)

torch.Size([4, 256])


In [40]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

torch.Size([8, 4, 256])


In [41]:
inputs = torch.tensor(
 [[0.43, 0.15, 0.89], # Your (x^1)
 [0.55, 0.87, 0.66], # journey (x^2)
 [0.57, 0.85, 0.64], # starts (x^3)
 [0.22, 0.58, 0.33], # with (x^4)
 [0.77, 0.25, 0.10], # one (x^5)
 [0.05, 0.80, 0.55]] # step (x^6)
)

query = inputs[1]
attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
 attn_scores_2[i] = torch.dot(x_i, query)
print(attn_scores_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


In [42]:
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()
print("Attention weights:", attn_weights_2_tmp)
print("Sum:", attn_weights_2_tmp.sum())

Attention weights: tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum: tensor(1.0000)


In [43]:
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)
attn_weights_2_naive = softmax_naive(attn_scores_2)
print("Attention weights:", attn_weights_2_naive)
print("Sum:", attn_weights_2_naive.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


In [44]:
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)
print("Attention weights:", attn_weights_2)
print("Sum:", attn_weights_2.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


In [45]:
query = inputs[1]
context_vec_2 = torch.zeros(query.shape)
for i,x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i]*x_i
print(context_vec_2)

tensor([0.4419, 0.6515, 0.5683])
